In [19]:
%run geo_zones

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 15, Finished, Available, Finished, True)

In [20]:
%run transformation

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 28, Finished, Available, Finished, True)

In [21]:
%run manage_tables

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 35, Finished, Available, Finished, True)

In [22]:
from pyspark.sql.functions import *

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 36, Finished, Available, Finished, False)

In [23]:
TABLE_BASE_NAME = "OpenAQ_measurements"
OPENAQ_TABLE_TYPES = ["hours", "days"]

OPENAQ_SCHEMA = {
    "station_id": {"type": "integer", "nullable": False, "primary_key": True},
    "sensor_id": {"type": "integer", "nullable": False, "primary_key": True},
    "parameter_id": {"type": "integer", "nullable": False, "primary_key": True},
    "parameter_name": {"type": "string", "nullable": False},
    "parameter_units": {"type": "string", "nullable": True},
    "parameter_display_name": {"type": "string", "nullable": True},
    "latitude": {"type": "double", "nullable": False},
    "longitude": {"type": "double", "nullable": False},
    "measurement_value": {"type": "double", "nullable": False},
    "measurement_datetime": {"type": "timestamp", "nullable": False, "primary_key": True}
}
OPENAQ_COLUMN_MAPPING = {
    "id": "station_id",
    "sensors_id": "sensor_id",
    "sensors_parameter_id": "parameter_id",
    "sensors_parameter_name": "parameter_name",
    "sensors_parameter_units": "parameter_units",
    "sensors_parameter_displayName": "parameter_display_name",
    "coordinates_latitude": "latitude",
    "coordinates_longitude": "longitude",
    "measurement_period_datetimeFrom_local": "measurement_datetime"
}

VALUE_RANGES = {
    "temperature": (-50, 60),
    "pm10": (0, 100),
    "pm25": (0, 100),
    "no2": (0, 0.1),
    "co": (0, 2.5),
    "so2": (0, 0.01),
    "o3": (0, 0.15),
    "pm1": (0, 100),
    "no": (0, 0.4),
    "relativehumidity": (0, 100),
    "um003": (0, 20000),
    "nox": (0, 0.7)
}

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 37, Finished, Available, Finished, False)

In [24]:
def remove_timezone(df, column):
    return df.withColumn(column, col(column).cast("string").substr(1, 19))

def filter_parameter_ranges(df, parameter_column, value_column, ranges):
    for parameter, (min_value, max_value) in ranges.items():
        df = df.filter((col(parameter_column) != parameter) | col(value_column).between(min_value, max_value))
    return df

def normalize_openaq(df, column_mapping, schema):
    df = normalize_column_names(df)
    df = rename_columns(df, column_mapping)
    df = select_columns(df, schema)
    df = remove_timezone(df, "measurement_datetime")
    df = cast_columns(df, {column: prop["type"] for column, prop in schema.items()})
    return df

def filter_openaq(df, schema):
    df = drop_null_rows(df, [column for column, prop in schema.items() if not prop["nullable"]])
    df = df.dropDuplicates([column for column, prop in schema.items() if prop.get("primary_key", False)])
    df = filter_rows(df, "measurement_flagInfo_hasFlags", False, "=")
    df = filter_parameter_ranges(df, "parameter_name", "measurement_value", VALUE_RANGES)
    return df

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 38, Finished, Available, Finished, False)

In [25]:
def read_openaq(table_type):
    table_name = f"{TABLE_BASE_NAME}_{table_type}"
    return read_table(table_name, "Bronze_lakehouse")

def load_and_process_openaq_table(table_type, column_mapping, shema):
    df = read_openaq(table_type)
    df = normalize_openaq(df, column_mapping, shema)
    df = filter_openaq(df, shema)
    return df

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 39, Finished, Available, Finished, False)

In [29]:
def process(table_type):
    sedona = prepare_spatial_processing(spark, TAXI_ZONES_SHAPEFILE_FOLDER_PATH)
    df = load_and_process_openaq_table(table_type, OPENAQ_COLUMN_MAPPING, OPENAQ_SCHEMA)
    df = add_zone_from_coordinates(df, sedona, "latitude", "longitude")
    return df

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 43, Finished, Available, Finished, False)

In [31]:
df = process("hours")

StatementMeta(, d11b0b87-8794-4247-9ce1-a3d03c38eed0, 45, Finished, Available, Finished, False)